# Dynamic Partition Pruning (DPP) in Spark
---------------------------------------------
Dynamic Partition Pruning is a Spark optimisation that:
Skips reading irrelevant partitions of a table during a join, using data discovered at runtime.
It’s especially powerful in large fact + dimension joins (star schema patterns).

Usually spark will :
Spark starts executing → learns join keys → skips unnecessary partitions of the big table

i.e, Fact table (big) 
sales
- 1 billion rows
- partitioned by country_id
Dimension table (small)
country
- 200 rows
Query : sales.join(country, "country_id")

Without DPP
Spark:
reads ALL sales partitions
even countries not in the join result

With Dynamic Partition Pruning
Spark:
reads country first
extracts relevant country_ids
filters sales partitions dynamically

In [0]:
from pyspark.sql.functions import *


In [0]:
epl_df = spark.read.table('sparkoptimization.tables.epl_final')
stadium_df = spark.read.table('sparkoptimization.tables.epl_stadiums')
epl_partitioned_df = spark.read.table('sparkoptimization.tables.epl_final_partitioned')

In [0]:
# epl_df.write.format('delta').mode('overwrite').partitionBy('HomeTeam').saveAsTable('sparkoptimization.tables.epl_final_partitioned')

# Turn OFF AQER and DDP and AutoBroadcast
----------------------------------------------

In [0]:
# ''' Usually these steps take place of of serverless compute'''
# spark.conf.set("spark.databricks.optimizer.dynamicPartitionPruning.enabled", "false")
# spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "False")
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [0]:
''' Dynamic partion purning in this example will take the stadiums table and make the filter there as well to speed up the query'''

epl_df.filter(col('HomeTeam') == 'Arsenal').join(stadium_df, epl_df.HomeTeam == stadium_df.Team, 'left').display()

In [0]:
epl_partitioned_df.filter(col('HomeTeam') == 'Arsenal').join(stadium_df, epl_partitioned_df.HomeTeam == stadium_df.Team, 'left').display()